In [ ]:
# Cell 1
# Mục đích: định nghĩa lại kết nối và hai hàm đo byte (notebook mới phải định nghĩa lại).
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")  # kênh kết nối, location khớp dataset

def mb(n):
    # đổi byte sang MB; trả None nếu n là None để lỗi đọc quá sớm lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # dry run: chỉ ước lượng byte, không chạy thật, không tốn tiền
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Se quet khoang: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # chạy thật, tắt cache, đặt trần byte billed; byte lấy từ job, kết quả nằm trong df
    cfg = bigquery.QueryJobConfig(use_query_cache=False, maximum_bytes_billed=max_mb * 1024 * 1024)
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

In [ ]:
# Cell 2
# Mục đích: kiểm tra dim_date đúng grain (1 dòng = 1 ngày) và phủ đủ khoảng ngày.
sql = f"""
select
  count(*) as so_dong,
  count(distinct date_day) as so_ngay,
  min(date_day) as tu_ngay,
  max(date_day) as den_ngay,
  countif(date_day is null) as ngay_null
from `{PROJECT}.dbt_dev.dim_date`
"""
check(sql)
df = stats(sql)
print(len(df))
df

Se quet Khoang: 0.03 MB
processed: 0.03 MB | billed: 10.0 MB 
1


,so_dong,so_ngay,tu_ngay,den_ngay,ngay_null
0,3287,3287,2019-01-01,2027-12-31,0


In [9]:
# Cell 3
# Mục đích: kiểm tra mọi order_date của fact đều có trong dim_date (join không làm mất ngày).
# Lưu ý: query đọc 2 bảng nên billed khoảng 20 MB (mức tối thiểu 10 MB mỗi bảng).
sql = f"""
select
  count(*) as so_dong_fact,
  countif(d.date_day is null) as ngay_khong_khop,
  min(f.order_date) as ngay_dau,
  max(f.order_date) as ngay_cuoi
from `{PROJECT}.dbt_dev.fct_net_sales` as f
left join `{PROJECT}.dbt_dev.dim_date` as d
  on f.order_date = d.date_day
"""
check(sql)
df = stats(sql)
df

Se quet Khoang: 0.37 MB
processed: 0.37 MB | billed: 20.0 MB 


,so_dong_fact,ngay_khong_khop,ngay_dau,ngay_cuoi
0,45529,0,2019-01-27,2026-10-14


In [8]:
# Cell 4
# Mục đích: lấy doanh thu theo năm từ SQL để đối chiếu với ma trận Power BI ở Bước 4.
sql = f"""
select
  d.year_number,
  count(*) as order_items,
  round(sum(f.sale_price), 2) as net_revenue
from `{PROJECT}.dbt_dev.fct_net_sales` as f
join `{PROJECT}.dbt_dev.dim_date` as d
  on f.order_date = d.date_day
group by d.year_number
order by d.year_number
"""
check(sql)
df = stats(sql)
print(len(df), df["net_revenue"].sum())
df

Se quet Khoang: 0.74 MB
processed: 0.74 MB | billed: 20.0 MB 
8 2685271.28


,year_number,order_items,net_revenue
0,2019,335,17837.11
1,2020,1212,71756.66
2,2021,2218,133658.87
3,2022,3459,202062.04
4,2023,4936,290075.70
5,2024,6929,406979.98
6,2025,10706,624117.64
7,2026,15734,938783.28
